# Air Traffic Data Analysis — Inferential Statistics and Regression

## Section 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

print("All libraries imported successfully.")

In [ ]:
import os

if os.path.exists("air_traffic_data.csv"):
    df = pd.read_csv("air_traffic_data.csv")
    print("Dataset loaded from file.")
else:
    # Generate sample data if CSV is not available
    np.random.seed(42)
    n = 200

    dom_flt = np.random.randint(400, 900, n)
    int_flt = np.random.randint(100, 400, n)
    dom_pax = dom_flt * np.random.uniform(80, 130, n) + np.random.normal(0, 5000, n)
    int_pax = int_flt * np.random.uniform(150, 250, n) + np.random.normal(0, 8000, n)
    dom_pax = dom_pax.clip(min=0).astype(int)
    int_pax = int_pax.clip(min=0).astype(int)

    df = pd.DataFrame({
        "Dom_Pax": dom_pax,
        "Int_Pax": int_pax,
        "Pax": dom_pax + int_pax,
        "Dom_Flt": dom_flt,
        "Int_Flt": int_flt,
        "Flt": dom_flt + int_flt,
        "Dom_RPM": dom_pax * np.random.uniform(500, 900, n)
    })

    print("CSV not found — sample data generated.")

print(f"Dataset shape: {df.shape}")
df.head()

## Section 2: Exploratory Data Analysis

In [ ]:
print("Dataset info:")
df.info()
print("\nFirst rows:")
print(df.head())
print("\nDescriptive statistics:")
print(df.describe())

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Correlation Matrix — Air Traffic Variables")
plt.tight_layout()
plt.show()

# Identify strongest correlations (excluding self-correlations)
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["Variable 1", "Variable 2", "Correlation"]
corr_pairs = corr_pairs.reindex(corr_pairs["Correlation"].abs().sort_values(ascending=False).index)

print("Strongest correlations (|r| > 0.7):")
print(corr_pairs[corr_pairs["Correlation"].abs() > 0.7].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
cols = ["Dom_Pax", "Int_Pax", "Pax", "Dom_Flt", "Int_Flt", "Flt"]

for ax, col in zip(axes.flatten(), cols):
    df[col].hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col)

plt.suptitle("Feature Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## Section 3: Hypothesis Testing

In [ ]:
alpha = 0.05

# --- Test 1: Domestic vs International Passengers ---
print("Test 1: Compare Domestic vs International Passengers")
print("H0: Mean domestic passengers = Mean international passengers")
print("H1: Mean domestic passengers != Mean international passengers")
print()

t_stat, p_value = stats.ttest_ind(df["Dom_Pax"], df["Int_Pax"])

print(f"Mean Dom_Pax : {df['Dom_Pax'].mean():,.0f}")
print(f"Mean Int_Pax : {df['Int_Pax'].mean():,.0f}")
print(f"T-statistic  : {t_stat:.4f}")
print(f"P-value      : {p_value:.6f}")
print()

if p_value < alpha:
    print(f"Result: p={p_value:.6f} < alpha={alpha} -> Reject H0.")
    print("There is a statistically significant difference between domestic and international passenger volumes.")
else:
    print(f"Result: p={p_value:.6f} >= alpha={alpha} -> Fail to reject H0.")
    print("No statistically significant difference between domestic and international passenger volumes.")

In [ ]:
# --- Test 2: Correlation between total passengers and total flights ---
print("Test 2: Correlation between Total Passengers (Pax) and Total Flights (Flt)")
print("H0: No correlation between Pax and Flt (rho = 0)")
print("H1: Significant correlation exists (rho != 0)")
print()

corr_coef, p_value_corr = stats.pearsonr(df["Pax"], df["Flt"])

print(f"Pearson r : {corr_coef:.4f}")
print(f"P-value   : {p_value_corr:.6f}")
print()

if p_value_corr < alpha:
    print(f"Result: p={p_value_corr:.6f} < alpha={alpha} -> Reject H0.")
    print(f"There is a statistically significant correlation (r={corr_coef:.4f}) between total passengers and total flights.")
else:
    print(f"Result: p={p_value_corr:.6f} >= alpha={alpha} -> Fail to reject H0.")
    print("No significant correlation found.")

plt.figure(figsize=(6, 4))
plt.scatter(df["Flt"], df["Pax"], alpha=0.5, color="steelblue", s=20)
plt.title(f"Total Flights vs Total Passengers (r = {corr_coef:.4f})")
plt.xlabel("Total Flights (Flt)")
plt.ylabel("Total Passengers (Pax)")
plt.tight_layout()
plt.show()

## Section 4: Simple Linear Regression

In [ ]:
# Predict total passengers (Pax) from total flights (Flt)
X_simple = df[["Flt"]]
y = df["Pax"]

X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

slr = LinearRegression()
slr.fit(X_train_s, y_train)
y_pred_s = slr.predict(X_test_s)

r2_s   = r2_score(y_test, y_pred_s)
mse_s  = mean_squared_error(y_test, y_pred_s)
rmse_s = np.sqrt(mse_s)
mae_s  = mean_absolute_error(y_test, y_pred_s)

print("Simple Linear Regression — Pax ~ Flt")
print("-" * 40)
print(f"Intercept  : {slr.intercept_:,.2f}")
print(f"Coefficient: {slr.coef_[0]:,.2f}")
print(f"R²         : {r2_s:.4f}")
print(f"MSE        : {mse_s:,.2f}")
print(f"RMSE       : {rmse_s:,.2f}")
print(f"MAE        : {mae_s:,.2f}")
print(f"\nModel equation: Pax = {slr.intercept_:,.0f} + {slr.coef_[0]:.2f} × Flt")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_s, alpha=0.5, color="steelblue", s=20)
min_val = min(y_test.min(), y_pred_s.min())
max_val = max(y_test.max(), y_pred_s.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=1.5)
axes[0].set_title(f"Actual vs Predicted (R²={r2_s:.4f})")
axes[0].set_xlabel("Actual Pax")
axes[0].set_ylabel("Predicted Pax")

# Residual plot
residuals_s = y_test - y_pred_s
axes[1].scatter(y_pred_s, residuals_s, alpha=0.5, color="tomato", s=20)
axes[1].axhline(0, color="black", lw=1.5, linestyle="--")
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted Pax")
axes[1].set_ylabel("Residuals")

plt.suptitle("Simple Linear Regression", fontsize=13)
plt.tight_layout()
plt.show()

## Section 5: Multiple Linear Regression

In [ ]:
# Features selected: Dom_Pax, Int_Pax, Dom_Flt, Int_Flt, Dom_RPM
# Excluded: Pax (target), Flt (sum of Dom_Flt + Int_Flt — would cause multicollinearity)
features = ["Dom_Pax", "Int_Pax", "Dom_Flt", "Int_Flt", "Dom_RPM"]
X_multi = df[features]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_m_sc = scaler.fit_transform(X_train_m)
X_test_m_sc  = scaler.transform(X_test_m)

mlr = LinearRegression()
mlr.fit(X_train_m_sc, y_train_m)
y_pred_m = mlr.predict(X_test_m_sc)

r2_m   = r2_score(y_test_m, y_pred_m)
mse_m  = mean_squared_error(y_test_m, y_pred_m)
rmse_m = np.sqrt(mse_m)
mae_m  = mean_absolute_error(y_test_m, y_pred_m)

print("Multiple Linear Regression")
print("-" * 40)
print(f"R²  : {r2_m:.4f}")
print(f"MSE : {mse_m:,.2f}")
print(f"RMSE: {rmse_m:,.2f}")
print(f"MAE : {mae_m:,.2f}")

print("\nFeature Coefficients (scaled):")
for feat, coef in zip(features, mlr.coef_):
    print(f"  {feat:15s}: {coef:,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test_m, y_pred_m, alpha=0.5, color="mediumseagreen", s=20)
min_val = min(y_test_m.min(), y_pred_m.min())
max_val = max(y_test_m.max(), y_pred_m.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=1.5)
axes[0].set_title(f"Actual vs Predicted (R²={r2_m:.4f})")
axes[0].set_xlabel("Actual Pax")
axes[0].set_ylabel("Predicted Pax")

residuals_m = y_test_m - y_pred_m
axes[1].scatter(y_pred_m, residuals_m, alpha=0.5, color="orchid", s=20)
axes[1].axhline(0, color="black", lw=1.5, linestyle="--")
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted Pax")
axes[1].set_ylabel("Residuals")

plt.suptitle("Multiple Linear Regression", fontsize=13)
plt.tight_layout()
plt.show()

# Feature importance bar chart
coef_df = pd.Series(np.abs(mlr.coef_), index=features).sort_values()
plt.figure(figsize=(7, 4))
coef_df.plot(kind="barh", color="steelblue")
plt.title("Feature Importance (|Coefficient|)")
plt.xlabel("Absolute Coefficient Value")
plt.tight_layout()
plt.show()

## Section 6: Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Simple Linear Regression",   "R²": r2_s, "RMSE": rmse_s, "MAE": mae_s},
    {"Model": "Multiple Linear Regression", "R²": r2_m, "RMSE": rmse_m, "MAE": mae_m}
])
print(comparison.to_string(index=False))

r2_improvement   = ((r2_m   - r2_s)   / r2_s)   * 100
rmse_improvement = ((rmse_s - rmse_m) / rmse_s) * 100
mae_improvement  = ((mae_s  - mae_m)  / mae_s)  * 100

print(f"\nR²   improvement  : {r2_improvement:+.2f}%")
print(f"RMSE improvement  : {rmse_improvement:+.2f}%")
print(f"MAE  improvement  : {mae_improvement:+.2f}%")

better = "Multiple Linear Regression" if r2_m > r2_s else "Simple Linear Regression"
print(f"\nBetter model: {better}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
models = ["Simple", "Multiple"]

axes[0].bar(models, [r2_s, r2_m], color=["steelblue", "mediumseagreen"])
axes[0].set_title("R² Score (higher is better)")
axes[0].set_ylim(0, 1)
for i, v in enumerate([r2_s, r2_m]):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center")

axes[1].bar(models, [rmse_s, rmse_m], color=["steelblue", "mediumseagreen"])
axes[1].set_title("RMSE (lower is better)")
for i, v in enumerate([rmse_s, rmse_m]):
    axes[1].text(i, v + rmse_s * 0.01, f"{v:,.0f}", ha="center")

axes[2].bar(models, [mae_s, mae_m], color=["steelblue", "mediumseagreen"])
axes[2].set_title("MAE (lower is better)")
for i, v in enumerate([mae_s, mae_m]):
    axes[2].text(i, v + mae_s * 0.01, f"{v:,.0f}", ha="center")

plt.suptitle("Model Comparison", fontsize=13)
plt.tight_layout()
plt.show()

## Section 7: Statistical Insights and Conclusions

**Hypothesis Test Results**

Test 1 compared domestic and international passenger volumes using an independent t-test. Given that domestic routes typically serve far more passengers than international routes, the test is expected to reject H₀, confirming a statistically significant difference between the two groups. This reflects a structural reality in air traffic: domestic networks are larger and more frequent.

Test 2 examined whether total passenger count and total number of flights are correlated. The Pearson test is expected to return a highly significant result (p << 0.05) with a strong positive correlation (r close to 1), since more flights directly increase the capacity to carry passengers. This correlation is both statistically and practically meaningful.

**Regression Model Performance**

The simple linear regression model uses only total flights to predict total passengers. While it captures a linear trend, it misses the nuance of how domestic and international travel behave differently. The multiple regression model, trained on five features including domestic/international breakdowns and revenue passenger-miles, provides a more complete picture and is expected to achieve a higher R² and lower error.

**Key Findings from Correlation Analysis**

The correlation matrix reveals strong positive relationships between passenger counts and flight counts across domestic and international segments. Dom_RPM is strongly correlated with Dom_Pax, which makes intuitive sense: more domestic passengers generate more revenue passenger-miles. The aggregate variables Pax and Flt are highly correlated with their component parts, confirming data consistency.

**Actionable Recommendations**

Airlines can use the regression models to forecast passenger demand from planned flight schedules. If the number of flights in a given period is known, the simple model offers a quick estimate. For more accurate capacity planning — including staffing, gate allocation, and fuel budgeting — the multiple regression model should be used, as it accounts for the domestic/international split and revenue miles. The strong correlation between Dom_RPM and domestic passengers also suggests that revenue performance is a reliable proxy for traffic volume, useful for financial forecasting.

## Section 8: Reflection Questions

**1. What do the hypothesis test results reveal about air traffic patterns?**

The t-test confirms that domestic and international passenger volumes are not interchangeable — they operate at different scales and serve different demand patterns. The Pearson test confirms that flight frequency is a strong predictor of passenger volume, which validates the core logic of airline capacity planning: more seats flown equals more passengers carried.

**2. Why did the multiple regression model perform better than the simple model?**

The simple model reduces all variation in passenger counts to a single variable (total flights), ignoring the structural differences between domestic and international travel. Domestic and international passengers have different average load factors, fare structures, and trip lengths. By including both segments separately, along with Dom_RPM, the multiple model captures more of the underlying variance in total passengers.

**3. How can airlines use correlation insights operationally?**

Strong correlations between passenger volumes and flight counts allow airlines to use flight schedules — which are planned months in advance — as leading indicators of passenger demand. Operations teams can align ground crew, baggage handling, and gate resources accordingly. The link between Dom_RPM and domestic passengers also helps revenue management teams track whether yield per passenger is evolving alongside traffic volumes.

**4. What do residual plots tell us about model assumptions?**

Residual plots should show randomly scattered points with no discernible pattern around the zero line. A systematic pattern (funnel shape, curve) would indicate that the model violates the assumption of homoscedasticity (constant variance of errors) or that the relationship is non-linear. If the residuals look randomly distributed, the linear regression assumptions are reasonably satisfied.

**5. What are practical applications of these statistical models?**

These models have direct applications in airline revenue forecasting, airport capacity planning, regulatory reporting, and network optimization. Airports use similar models to plan terminal expansions. Airlines use them to set flight frequencies on new routes. Civil aviation authorities use passenger and flight data to monitor industry health and project infrastructure needs.